# Selective Identity Association After Object Absence

This notebook runs the research workflow in Google Colab. It tests whether logistic-probability association and selective rejection reduce identity switches when an object reappears after a mask-defined absence. The masks do not distinguish occlusion from leaving the view.

**Scope:** DAVIS ground-truth instance masks are used as detections. This isolates association from segmentation errors; this project does not evaluate segmentation, object discovery, or general-purpose multi-object tracking.

Upload or clone this repository to `/content/occlusion-aware-object-association`, then download DAVIS 2017 TrainVal 480p from the [official DAVIS page](https://davischallenge.org/davis2017/code.html) into Google Drive. The default path is `/content/drive/MyDrive/DAVIS`. Both official train and validation split files are required.

## 1. Project introduction and research question

The unweighted logistic model estimates **P(correct association | association features)** from training-sequence candidate pairs. Below-threshold edges are excluded before one-to-one assignment, which includes unmatched identity/detection choices; unmatched candidate detections start new predicted identities. Validation sequences remain held out for final evaluation. Brier score and ECE assess calibration on held-out candidate pairs.

## 2. Check Python and CUDA environment

In [1]:
import platform
import sys

print("Python:", platform.python_version())
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA version:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        free_bytes, total_bytes = torch.cuda.mem_get_info(0)
        print(f"Available GPU memory: {free_bytes / (1024**3):.2f} GiB free / {total_bytes / (1024**3):.2f} GiB total")
    else:
        print("GPU: none; supported feature extraction falls back to CPU")
except ImportError:
    print("PyTorch is not installed in this runtime yet")

Python: 3.13.16
PyTorch: 2.14.1+cu130
CUDA version: 13.0
CUDA available: True
GPU: Tesla T4
Available GPU memory: 14.46 GiB free / 14.56 GiB total


In [2]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Arian-jafari/occlusion-aware-object-centric-learning.git"
PROJECT_DIR = Path("/content/occlusion-aware-object-association")

subprocess.run(
    [
        "git", "clone",
        "--branch", "research-readiness-corrections",
        REPO_URL,
        str(PROJECT_DIR)
    ],
    check=True
)

print("Repository cloned successfully.")

Repository cloned successfully.


In [3]:
%cd /content/occlusion-aware-object-association

!git branch --show-current
!git log -1 --oneline
!ls

/content/occlusion-aware-object-association
research-readiness-corrections
7a63f3c (HEAD -> research-readiness-corrections, origin/research-readiness-corrections) Add threshold-specific tracking evaluation
CITATION.cff  docs     notebooks       README.md	 results  src
configs       LICENSE  pyproject.toml  requirements.txt  scripts  tests


## 3. Install dependencies

In [4]:
from pathlib import Path

PROJECT_DIR = Path("/content/occlusion-aware-object-association")
if not (PROJECT_DIR / "requirements.txt").is_file():
    raise FileNotFoundError(
        f"{PROJECT_DIR} does not contain requirements.txt. Upload or clone the repository there first."
    )
%pip install -q -r /content/occlusion-aware-object-association/requirements.txt
%pip install -q -e /content/occlusion-aware-object-association

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for occlusion-association (pyproject.toml) ... done


## 4. Mount Google Drive

In [5]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
!python -m pytest -q

..........................................F.......                       [100%]
=================================== FAILURES ===================================
______ test_sequence_cache_invalidates_source_contents_and_model_revision ______

tmp_path = PosixPath('/tmp/pytest-of-root/pytest-0/test_sequence_cache_invalidate0')

    def test_sequence_cache_invalidates_source_contents_and_model_revision(tmp_path: Path) -> None:
        image = Image.new("RGB", (14, 14), (40, 80, 120))
        mask = np.zeros((14, 14), dtype=np.uint8)
        mask[:, :7] = 1
        mask[:, 7:] = 2
        frame = FrameData(
            sequence="seq",
            frame_name="00000",
            image=image,
            mask=mask,
            object_ids=(1, 2),
            image_path=tmp_path / "00000.jpg",
            annotation_path=tmp_path / "00000.png",
        )
        sequence = SequenceData(name="seq", frames=[frame])
        extractor = _FakeSpatialExtractor()
        cache = FeatureCache(tmp_pat

In [7]:
import shutil
from pathlib import Path
from tqdm.auto import tqdm

DRIVE_DATASET = Path("/content/drive/MyDrive/DAVIS")
LOCAL_DATASET = Path("/content/DAVIS")

if not DRIVE_DATASET.exists():
    raise FileNotFoundError(f"DAVIS dataset not found: {DRIVE_DATASET}")

if LOCAL_DATASET.exists():
    print("Local DAVIS dataset already exists.")
    print("Skipping copy to avoid unnecessary file transfers.")

else:
    print("Counting dataset files...")

    all_files = [
        path for path in DRIVE_DATASET.rglob("*")
        if path.is_file()
    ]

    total_files = len(all_files)

    print(f"Files to copy: {total_files:,}")

    LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

    with tqdm(
        total=total_files,
        desc="Copying DAVIS",
        unit="file",
        dynamic_ncols=True,
    ) as pbar:

        for source_file in all_files:
            relative_path = source_file.relative_to(DRIVE_DATASET)
            destination_file = LOCAL_DATASET / relative_path

            destination_file.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            shutil.copy2(source_file, destination_file)

            pbar.update(1)

    print("\nDAVIS dataset copied successfully!")

print("Dataset location:", LOCAL_DATASET)

Counting dataset files...
Files to copy: 12,422


Copying DAVIS:   0%|          | 0/12422 [00:00<?, ?file/s]


DAVIS dataset copied successfully!
Dataset location: /content/DAVIS


## 5. Configure dataset and run mode

In [8]:
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

from occlusion_association.config import load_config

DAVIS_ROOT = Path("/content/drive/MyDrive/DAVIS")  # Change only this value if your Drive layout differs.
RUN_MODE = "full"  # Choose "smoke" for a small run or "full" for the official train/validation split.
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

config = load_config(
    PROJECT_DIR / "configs" / "default.yaml",
    overrides={
        "dataset": {"root": str(DAVIS_ROOT)},
        "experiment": {"smoke_test": RUN_MODE == "smoke"},
    },
)
print("Repository:", PROJECT_DIR)
print("DAVIS root:", config.dataset_root)
print("Run mode:", RUN_MODE)
print("Persistent feature cache:", config.features["cache_dir"])
print("Persistent experiment output:", config.experiment["output_dir"])

Repository: /content/occlusion-aware-object-association
DAVIS root: /content/drive/MyDrive/DAVIS
Run mode: full
Persistent feature cache: /content/drive/MyDrive/occlusion_aware_association/cache/features
Persistent experiment output: /content/drive/MyDrive/occlusion_aware_association/results


## 6. Validate DAVIS dataset structure

In [9]:
from occlusion_association.dataset import DAVISDataset

dataset = DAVISDataset(
    config.dataset_root,
    split="all",
    train_split=config.dataset["train_split"],
    validation_split=config.dataset["validation_split"],
)
print("Resolved DAVIS root:", dataset.root)
print("Official train sequences:", len(dataset.train_sequences))
print("Official validation sequences:", len(dataset.validation_sequences))
print("Split files:", config.dataset["train_split"], "and", config.dataset["validation_split"])

Resolved DAVIS root: /content/drive/MyDrive/DAVIS
Official train sequences: 60
Official validation sequences: 30
Split files: train and val


## 7. Inspect dataset

In [10]:
from tqdm.auto import tqdm
from occlusion_association.utils import select_smoke_sequence_names

if RUN_MODE == "smoke":
    train_sequence_names = select_smoke_sequence_names(
        dataset, dataset.train_sequences, int(config.experiment["smoke_test_sequences"])
    )
    validation_sequence_names = select_smoke_sequence_names(
        dataset, dataset.validation_sequences, int(config.experiment["smoke_test_sequences"])
    )
    FRAME_LIMIT = int(config.experiment["smoke_test_frames"])
else:
    train_sequence_names = list(dataset.train_sequences)
    validation_sequence_names = list(dataset.validation_sequences)
    FRAME_LIMIT = None

selected_names = train_sequence_names + validation_sequence_names

print(f"Run mode: {RUN_MODE}")
print(f"Training sequences: {len(train_sequence_names)}")
print(f"Validation sequences: {len(validation_sequence_names)}")
print(f"Total sequences to inspect: {len(selected_names)}")
print(f"Frame limit: {FRAME_LIMIT or 'All frames'}")

# Inspect each sequence separately to expose progress.
inspected_sequences = []
loading_errors = []

for sequence_name in tqdm(
    selected_names,
    desc="Inspecting DAVIS sequences",
    unit="sequence",
):
    result = dataset.inspect(
        sequence_names=[sequence_name],
        frame_limit=FRAME_LIMIT,
    )

    inspected_sequences.extend(result.inspected_sequences)
    loading_errors.extend(result.loading_errors)

# Print inspection results.
for sequence in inspected_sequences:
    status = "SKIPPED" if sequence.skipped else "ok"

    print(
        f"{sequence.sequence}: {status}; valid frames={sequence.valid_frames}/"
        f"{sequence.matched_frames} matched; objects={sequence.unique_objects}; "
        f"image sizes={sequence.image_dimensions}; "
        f"annotation sizes={sequence.annotation_dimensions}"
    )

    for lifecycle in sequence.object_lifecycles:
        print(
            f"  object {lifecycle['object_id']}: "
            f"first={lifecycle['first_observed_frame']}, "
            f"last={lifecycle['last_observed_frame']}, "
            f"valid-frame gaps={len(lifecycle['missing_gaps'])}"
        )

print("Dataset loading issues:", len(loading_errors))

for issue in loading_errors:
    print(
        f"[{issue.severity}:{issue.code}] "
        f"{issue.sequence or ''} {issue.frame or ''}: {issue.message}"
    )

Run mode: full
Training sequences: 60
Validation sequences: 30
Total sequences to inspect: 90
Frame limit: All frames


Inspecting DAVIS sequences:   0%|          | 0/90 [00:00<?, ?sequence/s]

bear: ok; valid frames=82/82 matched; objects=1; image sizes=[(854, 480)]; annotation sizes=[(854, 480)]
  object 1: first=00000, last=00081, valid-frame gaps=0
bmx-bumps: ok; valid frames=90/90 matched; objects=2; image sizes=[(854, 480)]; annotation sizes=[(854, 480)]
  object 1: first=00000, last=00089, valid-frame gaps=3
  object 2: first=00000, last=00089, valid-frame gaps=2
boat: ok; valid frames=75/75 matched; objects=1; image sizes=[(854, 480)]; annotation sizes=[(854, 480)]
  object 1: first=00000, last=00074, valid-frame gaps=0
boxing-fisheye: ok; valid frames=87/87 matched; objects=3; image sizes=[(854, 480)]; annotation sizes=[(854, 480)]
  object 1: first=00000, last=00086, valid-frame gaps=0
  object 2: first=00000, last=00086, valid-frame gaps=0
  object 3: first=00000, last=00086, valid-frame gaps=0
breakdance-flare: ok; valid frames=71/71 matched; objects=1; image sizes=[(854, 480)]; annotation sizes=[(854, 480)]
  object 1: first=00000, last=00070, valid-frame gaps=0


## 8. Load frozen DINOv2 ViT-S/14

In [11]:
from occlusion_association.features import DINOv2FeatureExtractor

extractor = DINOv2FeatureExtractor(
    backbone_name=config.features["backbone"],
    repository=config.features["torch_hub_repository"],
    input_size=int(config.features["input_size"]),
    device=config.features["device"],
    use_mixed_precision=bool(config.features["use_mixed_precision"]),
)
print("Frozen backbone:", extractor.backbone_name)
print("Device:", extractor.device)
print("Preprocessing/cache identity:", extractor.preprocessing)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 363MB/s]


Frozen backbone: facebookresearch/dinov2:dinov2_vits14
Device: cuda
Preprocessing/cache identity: {'input_size': 518, 'resize_policy': 'preserve_aspect_round_dimensions_to_patch_multiple', 'interpolation': 'bicubic', 'patch_size': 14, 'normalization_mean': [0.485, 0.456, 0.406], 'normalization_std': [0.229, 0.224, 0.225], 'device_type': 'cuda', 'precision_mode': 'cuda_fp16'}


## 9. Extract mask-pooled object features

In [13]:
from tqdm.auto import tqdm

from occlusion_association.feature_cache import FeatureCache
from occlusion_association.features import extract_dataset_features

cache = FeatureCache(config.features["cache_dir"])

selected_names = train_sequence_names + validation_sequence_names

feature_summaries = []
feature_issues = []

print(f"Run mode: {RUN_MODE}")
print(f"Sequences: {len(selected_names)}")
print(f"Batch size: {config.features['batch_size']}")
print(f"Frame limit: {FRAME_LIMIT or 'All frames'}")
print(f"Feature cache: {config.features['cache_dir']}")

# Extract features one sequence at a time to expose progress.
with tqdm(
    total=len(selected_names),
    desc="Extracting DINOv2 features",
    unit="sequence",
    dynamic_ncols=True,
) as pbar:

    for sequence_name in selected_names:

        pbar.set_postfix_str(f"Current: {sequence_name}")

        summaries, issues = extract_dataset_features(
            dataset,
            [sequence_name],
            extractor,
            cache,
            batch_size=int(config.features["batch_size"]),
            frame_limit=FRAME_LIMIT,
        )

        feature_summaries.extend(summaries)
        feature_issues.extend(issues)

        pbar.update(1)

# Display extraction summaries.
print("\nFeature extraction summaries:")

for summary in feature_summaries:
    print(summary)

# Display loading issues.
print("\nFeature extraction loading issues:", len(feature_issues))

for issue in feature_issues:
    print(
        f"[{issue.severity}:{issue.code}] "
        f"{issue.sequence or ''} {issue.frame or ''}: {issue.message}"
    )

Run mode: full
Sequences: 90
Batch size: 4
Frame limit: All frames
Feature cache: /content/drive/MyDrive/occlusion_aware_association/cache/features


Extracting DINOv2 features:   0%|          | 0/90 [00:00<?, ?sequence/s]


Feature extraction summaries:
FeatureExtractionSummary(sequence='bear', objects_requested=82, cache_hits=82, features_written=0, invalid_objects=[])
FeatureExtractionSummary(sequence='bmx-bumps', objects_requested=133, cache_hits=133, features_written=0, invalid_objects=[])
FeatureExtractionSummary(sequence='boat', objects_requested=75, cache_hits=75, features_written=0, invalid_objects=[])
FeatureExtractionSummary(sequence='boxing-fisheye', objects_requested=261, cache_hits=261, features_written=0, invalid_objects=[])
FeatureExtractionSummary(sequence='breakdance-flare', objects_requested=71, cache_hits=71, features_written=0, invalid_objects=[])
FeatureExtractionSummary(sequence='bus', objects_requested=80, cache_hits=80, features_written=0, invalid_objects=[])
FeatureExtractionSummary(sequence='car-turn', objects_requested=80, cache_hits=80, features_written=0, invalid_objects=[])
FeatureExtractionSummary(sequence='cat-girl', objects_requested=178, cache_hits=178, features_written=

## 10. Cache and verify features

In [15]:
from itertools import islice
from tqdm.auto import tqdm

from occlusion_association.features import load_cached_frame_embeddings

cache_check = {
    "frames": 0,
    "object_features": 0,
    "missing_features": 0,
}

cache_issues = []

with tqdm(
    total=len(selected_names),
    desc="Verifying feature cache",
    unit="sequence",
    dynamic_ncols=True,
) as sequence_bar:

    for sequence_name in selected_names:

        sequence_bar.set_postfix_str(f"Current: {sequence_name}")

        frame_iterator = dataset.iter_frames(
            sequence_name,
            issue_sink=cache_issues,
        )

        if FRAME_LIMIT is not None:
            frame_iterator = islice(frame_iterator, FRAME_LIMIT)

        # Count frames for this sequence to provide an accurate progress bar.
        frames = list(frame_iterator)

        for frame in tqdm(
            frames,
            desc=f"  {sequence_name}",
            unit="frame",
            leave=False,
            dynamic_ncols=True,
        ):
            embeddings, missing = load_cached_frame_embeddings(
                frame,
                cache,
                backbone_name=extractor.backbone_name,
                preprocessing=extractor.preprocessing,
            )

            cache_check["frames"] += 1
            cache_check["object_features"] += len(embeddings)
            cache_check["missing_features"] += len(missing)

        sequence_bar.update(1)

print("\nFeature cache verification:")
print(cache_check)

print("Cache loading issues:", len(cache_issues))

for issue in cache_issues:
    print(
        f"[{issue.severity}:{issue.code}] "
        f"{issue.sequence or ''} {issue.frame or ''}: {issue.message}"
    )

if cache_check["missing_features"]:
    raise RuntimeError(
        "Feature cache verification failed; "
        "rerun extraction before calibration."
    )

print("Feature cache verification successful!")

Verifying feature cache:   0%|          | 0/90 [00:00<?, ?sequence/s]

  bear:   0%|          | 0/82 [00:00<?, ?frame/s]

  bmx-bumps:   0%|          | 0/90 [00:00<?, ?frame/s]

  boat:   0%|          | 0/75 [00:00<?, ?frame/s]

  boxing-fisheye:   0%|          | 0/87 [00:00<?, ?frame/s]

  breakdance-flare:   0%|          | 0/71 [00:00<?, ?frame/s]

... (82 outputs left out)

  scooter-black:   0%|          | 0/43 [00:00<?, ?frame/s]

  shooting:   0%|          | 0/40 [00:00<?, ?frame/s]

  soapbox:   0%|          | 0/99 [00:00<?, ?frame/s]


Feature cache verification:
{'frames': 6208, 'object_features': 13556, 'missing_features': 0}
Cache loading issues: 0
Feature cache verification successful!


## 11. Generate training association examples

In [17]:
from itertools import islice
from tqdm.auto import tqdm

from occlusion_association.calibration import generate_calibration_candidates


def training_streams():
    for sequence_name in tqdm(
        train_sequence_names,
        desc="Preparing calibration sequences",
        unit="sequence",
        dynamic_ncols=True,
    ):
        frames = dataset.iter_frames(sequence_name)

        if FRAME_LIMIT is not None:
            frames = islice(frames, FRAME_LIMIT)

        yield sequence_name, frames


print("Generating calibration candidates...")
print("Training sequences:", len(train_sequence_names))
print("Maximum memory gap:", config.memory["max_gap"])
print("Appearance history:", config.memory["appearance_history_size"])

training_candidates = generate_calibration_candidates(
    training_streams(),
    cache,
    backbone_name=extractor.backbone_name,
    preprocessing=extractor.preprocessing,
    max_gap=int(config.memory["max_gap"]),
    appearance_history_size=int(config.memory["appearance_history_size"]),
)

print("\nCalibration candidate generation completed.")

print(
    "Training sequence names:",
    sorted(training_candidates.sequence_names),
)

print(
    "Candidate matrix:",
    training_candidates.features.shape,
)

print(
    "Positive/negative targets:",
    int((training_candidates.labels == 1).sum()),
    int((training_candidates.labels == 0).sum()),
)

Generating calibration candidates...
Training sequences: 60
Maximum memory gap: 30
Appearance history: 5


Preparing calibration sequences:   0%|          | 0/60 [00:00<?, ?sequence/s]


Calibration candidate generation completed.
Training sequence names: ['bear', 'bmx-bumps', 'boat', 'boxing-fisheye', 'breakdance-flare', 'bus', 'car-turn', 'cat-girl', 'classic-car', 'color-run', 'crossing', 'dance-jump', 'dancing', 'disc-jockey', 'dog-agility', 'dog-gooses', 'dogs-scale', 'drift-turn', 'drone', 'elephant', 'flamingo', 'hike', 'hockey', 'horsejump-low', 'kid-football', 'kite-walk', 'koala', 'lady-running', 'lindy-hop', 'longboard', 'lucia', 'mallard-fly', 'mallard-water', 'miami-surf', 'motocross-bumps', 'motorbike', 'night-race', 'paragliding', 'planes-water', 'rallye', 'rhino', 'rollerblade', 'schoolgirls', 'scooter-board', 'scooter-gray', 'sheep', 'skate-park', 'snowboard', 'soccerball', 'stroller', 'stunt', 'surf', 'swing', 'tennis', 'tractor-sand', 'train', 'tuk-tuk', 'upside-down', 'varanus-cage', 'walking']
Candidate matrix: (32644, 6)
Positive/negative targets: 9750 22894


## 12. Fit the confidence calibrator

In [18]:
from occlusion_association.calibration import fit_logistic_calibrator

calibrator = fit_logistic_calibrator(
    training_candidates,
    training_sequences=train_sequence_names,
    validation_sequences=validation_sequence_names,
    class_weight=config.calibration["class_weight"],
    max_iter=int(config.calibration["max_iter"]),
    random_state=int(config.calibration["random_state"]),
)
calibrator.metadata["validation_sequences_checked"] = validation_sequence_names
calibrator.metadata["max_memory_gap"] = int(config.memory["max_gap"])
print(calibrator.metadata)

{'model': 'standardized_logistic_regression', 'feature_names': ['cosine_similarity', 'center_displacement', 'bbox_iou', 'area_ratio', 'frames_since_last_observation', 'similarity_margin'], 'training_sequences': ['bear', 'bmx-bumps', 'boat', 'boxing-fisheye', 'breakdance-flare', 'bus', 'car-turn', 'cat-girl', 'classic-car', 'color-run', 'crossing', 'dance-jump', 'dancing', 'disc-jockey', 'dog-agility', 'dog-gooses', 'dogs-scale', 'drift-turn', 'drone', 'elephant', 'flamingo', 'hike', 'hockey', 'horsejump-low', 'kid-football', 'kite-walk', 'koala', 'lady-running', 'lindy-hop', 'longboard', 'lucia', 'mallard-fly', 'mallard-water', 'miami-surf', 'motocross-bumps', 'motorbike', 'night-race', 'paragliding', 'planes-water', 'rallye', 'rhino', 'rollerblade', 'schoolgirls', 'scooter-board', 'scooter-gray', 'sheep', 'skate-park', 'snowboard', 'soccerball', 'stroller', 'stunt', 'surf', 'swing', 'tennis', 'tractor-sand', 'train', 'tuk-tuk', 'upside-down', 'varanus-cage', 'walking'], 'validation_se

## 13. Run the association baselines

In [ ]:
from occlusion_association.experiments import run_experiment_suite

# The runner applies identical oracle-mask detections and cached frozen features to all methods.
# It also performs the proposed method and the requested ablations in the same held-out run.
experiment_results = run_experiment_suite(
    dataset,
    cache,
    config.to_dict(),
    train_sequence_names=train_sequence_names,
    validation_sequence_names=validation_sequence_names,
    frame_limit=FRAME_LIMIT,
    training_candidates=training_candidates,
    fitted_calibrator=calibrator,
)
experiment_results["smoke_test"] = RUN_MODE == "smoke"
import pandas as pd
pd.DataFrame(
    [row for row in experiment_results["metrics"] if row["method"] != "calibrated_selective"]
)

## 14. Run the proposed selective method

In [ ]:
pd.DataFrame(
    [row for row in experiment_results["metrics"] if row["method"] == "calibrated_selective"]
)

## 15. Run ablations

In [ ]:
print("Appearance versus motion")
display(pd.DataFrame(experiment_results["appearance_ablation"]))
print("Score-method comparison (feature sets differ; this does not isolate calibration)")
display(pd.DataFrame(experiment_results["calibration_ablation"]))
print("Memory duration")
display(pd.DataFrame(experiment_results["memory_duration_ablation"]))
print("Full-tracking results by confidence threshold")
display(pd.DataFrame(experiment_results["tracking_confidence_threshold_metrics"]))

## 16. Calculate and inspect metrics

In [ ]:
print("Held-out candidate-pair calibration metrics")
display(pd.DataFrame([{
    key: value for key, value in experiment_results["calibration_metrics"].items()
    if key not in {"reliability"}
}]))
print("Per-sequence tracking and association metrics")
display(pd.DataFrame(experiment_results["per_sequence_metrics"]).drop(
    columns=["occlusion_gaps", "candidate_labels", "candidate_probabilities"],
    errors="ignore",
))
print("Natural object absence/reappearance analysis")
display(pd.DataFrame([
    {"method": method, **point}
    for method, points in experiment_results["occlusion_gap_metrics"].items()
    for point in points
]))

## 17. Generate publication-style plots

In [ ]:
from occlusion_association.result_io import generate_plots_from_results

preview_directory = Path("/tmp/occlusion_association_plot_preview")
preview_figures = generate_plots_from_results(experiment_results, preview_directory)
print("Preview figures:", len(preview_figures))
print("PNG and PDF preview files:", preview_figures[:6])

## 18. Export results

In [22]:
from occlusion_association.result_io import export_experiment_results
from occlusion_association.utils import create_experiment_directory, runtime_metadata, set_random_seeds

set_random_seeds(int(config.experiment["random_seed"]))
experiment_directory = create_experiment_directory(config.experiment["output_dir"])
metadata = runtime_metadata(PROJECT_DIR)
metadata.update({
    "dataset_root": str(dataset.root),
    "dataset_split": {
        "train": train_sequence_names,
        "validation": validation_sequence_names,
    },
    "backbone": extractor.backbone_name,
    "random_seed": int(config.experiment["random_seed"]),
    "smoke_test": RUN_MODE == "smoke",
    "frame_limit": FRAME_LIMIT,
    "feature_cache_directory": str(cache.directory),
    "experiment_parameters": config.to_dict(),
})
(experiment_directory / "logs").mkdir(parents=True, exist_ok=True)
(experiment_directory / "logs" / "run.log").write_text(
    "Completed via notebooks/occlusion_aware_association.ipynb\n",
    encoding="utf-8",
)
export_experiment_results(
    experiment_results,
    experiment_directory,
    config=config.to_dict(),
    metadata=metadata,
    make_plots=True,
)
print("Exported to:", experiment_directory)

NameError: name 'experiment_results' is not defined

## 19. Generate experiment summary

In [ ]:
summary_columns = [
    "method",
    "identity_switches",
    "track_fragmentations",
    "association_precision",
    "association_recall",
    "association_coverage",
    "false_association_rate",
    "brier_score",
    "ece",
]
display(pd.DataFrame(experiment_results["metrics"])[summary_columns])
print("Calibrator training sequences:", experiment_results["train_candidate_summary"]["sequences"])
print("Validation sequences:", experiment_results["sequence_split"]["validation"])
print("Smoke test active:", experiment_results["smoke_test"])
print("Result directory:", experiment_directory)
print(
    "Limitations: ground-truth masks are detections; segmentation is not evaluated. "
    "Absence analysis uses only intervals with complete valid annotations; the mask does not identify why an object was absent."
)